In [6]:
import pandas as pd
import yaml
from pathlib import Path

In [8]:
def load_rules(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)["attacks"]

def convert_timestamps(df):
    # Zeek ts is Unix epoch in UTC
    df["datetime"] = pd.to_datetime(df["ts"], unit="s", utc=True)
    df["datetime"] = df["datetime"].dt.tz_convert("Canada/Atlantic")
    df["time"] = df["datetime"].dt.time
    return df

def label_flows_vectorized(df, rules):
    df["label"] = "BENIGN"

    for rule in rules:
        start = pd.to_datetime(rule["start"]).time()
        end = pd.to_datetime(rule["end"]).time()

        attacker_ips = rule["attacker_ips"]
        victim_ips = rule["victim_ips"]

        mask = (
            (df["time"] >= start) &
            (df["time"] <= end) &
            (
                (
                    df["id.orig_h"].isin(attacker_ips) &
                    df["id.resp_h"].isin(victim_ips)
                )
                |
                (
                    df["id.orig_h"].isin(victim_ips) &
                    df["id.resp_h"].isin(attacker_ips)
                )
            )
        )

        df.loc[mask, "label"] = rule["label"]
        print(f'{rule["name"]}: {mask.sum()} matches')

    return df

def load_and_merge_conn_files(conn_files):
    dfs = []

    for conn_path in conn_files:
        conn_path = Path(conn_path)
        print(f"Loading: {conn_path}")

        df = pd.read_csv(conn_path, sep="\t")
        dfs.append(df)

    merged = pd.concat(dfs, ignore_index=True)
    return merged

def generate_day_dataset(conn_files, rules_path, output_path):
    df = load_and_merge_conn_files(conn_files)
    df = convert_timestamps(df)

    print("\nDataset time range:")
    print(df["time"].min(), "->", df["time"].max())

    rules = load_rules(rules_path)

    print("\nApplying labels...")
    df = label_flows_vectorized(df, rules)

    print("\nLabel distribution:")
    print(df["label"].value_counts())

    df.drop(columns=["datetime", "time"], inplace=True)
    df.to_csv(output_path, sep="\t", index=False)
    print(f"\nSaved: {output_path}")

In [10]:
first_day_files_1 = sorted(Path("zeek_logs/PCAP-01-12_0-0249").glob("*/conn.tsv"))

generate_day_dataset(
    conn_files=first_day_files_1,
    rules_path="rules/first_day.yaml",
    output_path="cicddos2019_first_day_1_labeled.tsv"
)

Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_01\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_010\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0100\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0101\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0102\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0103\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0104\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0105\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0106\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0107\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0108\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0109\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_011\conn.tsv
Loading: zeek_logs\PCAP-01-12_0-0249\SAT-01-12-2018_0110\conn.tsv
Loading: zeek_log

In [4]:
first_day_files_2 = sorted(Path("zeek_logs/PCAP-01-12_0250-0499").glob("*/conn.tsv"))

generate_day_dataset(
    conn_files=first_day_files_2,
    rules_path="rules/first_day.yaml",
    output_path="cicddos2019_first_day_2_labeled.tsv"
)

Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0250\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0251\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0252\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0253\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0254\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0255\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0256\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0257\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0258\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0259\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0260\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0261\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0262\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250-0499\SAT-01-12-2018_0263\conn.tsv
Loading: zeek_logs\PCAP-01-12_0250

In [5]:
first_day_files_3 = sorted(Path("zeek_logs/PCAP-01-12_0500-0749").glob("*/conn.tsv"))

generate_day_dataset(
    conn_files=first_day_files_3,
    rules_path="rules/first_day.yaml",
    output_path="cicddos2019_first_day_3_labeled.tsv"
)

Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0500\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0501\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0502\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0503\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0504\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0505\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0506\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0507\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0508\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0509\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0510\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0511\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0512\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0513\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500

C:\Users\Rasmus\AppData\Local\Temp\ipykernel_12604\2533923480.py:50: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(conn_path, sep="\t")


Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0616\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0617\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0618\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0619\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0620\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0621\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0622\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0623\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0624\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0625\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0626\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0627\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0628\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500-0749\SAT-01-12-2018_0629\conn.tsv
Loading: zeek_logs\PCAP-01-12_0500

In [3]:
first_day_files_4 = sorted(Path("zeek_logs/PCAP-01-12_0750-0818").glob("*/conn.tsv"))

generate_day_dataset(
    conn_files=first_day_files_4,
    rules_path="rules/first_day.yaml",
    output_path="cicddos2019_first_day_4_labeled.tsv"
)

Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0750\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0751\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0752\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0753\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0754\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0755\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0756\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0757\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0758\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0759\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0760\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0761\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0762\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0763\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750

C:\Users\Rasmus\AppData\Local\Temp\ipykernel_12604\2533923480.py:50: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(conn_path, sep="\t")


Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0770\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0771\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0772\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0773\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0774\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0775\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0776\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0777\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0778\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0779\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0780\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0781\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0782\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750-0818\SAT-01-12-2018_0783\conn.tsv
Loading: zeek_logs\PCAP-01-12_0750

In [5]:
second_day_files = sorted(Path("zeek_logs/PCAP-03-11").glob("*/conn.tsv"))

generate_day_dataset(
    conn_files=second_day_files,
    rules_path="rules/second_day.yaml",
    output_path="cicddos2019_second_day_labeled.tsv"
)

Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_01\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_010\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0100\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0101\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0102\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0103\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0104\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0105\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0106\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0107\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0108\conn.tsv


C:\Users\Rasmus\AppData\Local\Temp\ipykernel_15524\2533923480.py:50: DtypeWarning: Columns (0: duration, 1: orig_bytes, 2: resp_bytes) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(conn_path, sep="\t")


Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0109\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_011\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0110\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0111\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0112\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0113\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0114\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0115\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0116\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0117\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0118\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0119\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_012\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0120\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0121\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0122\conn.tsv
Loading: zeek_logs\PCAP-03-11\SAT-03-11-2018_0123\conn.tsv